In [9]:
# 如需重新安装依赖库，可取消下方注释运行：
# %pip install mediapipe ultralytics opencv-python matplotlib --user

import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO

In [10]:
# 1. 加载 YOLOv8 目标检测模型
yolo_model = YOLO('yolov8n.pt')

# 2. 初始化 MediaPipe Task Vision PoseLandmarker
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_heavy.task'),
    running_mode=VisionRunningMode.VIDEO,
    min_pose_detection_confidence=0.5
)

# 3. 定义人体躯干与四肢骨骼连接规则 (排除面部 0-10 号节点)
body_connections = [
    (11, 12), (11, 23), (12, 24), (23, 24),            # 躯干
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),  # 左臂
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),  # 右臂
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),  # 左腿
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32)   # 右腿
]
body_landmark_indices = set(range(11, 33))

In [11]:
def process_badminton_video(input_path, output_path, yolo_model, options):
    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"🎬 开始处理视频: {input_path}")

    with PoseLandmarker.create_from_options(options) as landmarker:
        frame_count = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            # A. YOLOv8 人体框检测
            yolo_results = yolo_model(frame, classes=[0], conf=0.5, verbose=False)
            yolo_boxes = []
            if len(yolo_results[0].boxes) > 0:
                for box in yolo_results[0].boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    yolo_boxes.append((x1, y1, x2, y2))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # B. 只有检测到目标人体时，才进行 MediaPipe 姿态提取与碰撞过滤
            if yolo_boxes:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                frame_timestamp_ms = int(frame_count * (1000 / fps))
                
                pose_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
                
                if pose_result.pose_landmarks:
                    for person_landmarks in pose_result.pose_landmarks:
                        points = {
                            i: (int(lm.x * width), int(lm.y * height)) 
                            for i, lm in enumerate(person_landmarks)
                        }
                        
                        hip_x = (points[23][0] + points[24][0]) // 2
                        hip_y = (points[23][1] + points[24][1]) // 2
                        
                        is_inside_yolo_box = any(
                            x1 <= hip_x <= x2 and y1 <= hip_y <= y2 
                            for x1, y1, x2, y2 in yolo_boxes
                        )
                        
                        if is_inside_yolo_box:
                            for p1, p2 in body_connections:
                                if p1 in points and p2 in points:
                                    cv2.line(frame, points[p1], points[p2], (255, 255, 0), 2)
                            for idx in body_landmark_indices:
                                if idx in points:
                                    cv2.circle(frame, points[idx], 4, (0, 0, 255), -1)

            out.write(frame)
            frame_count += 1

    cap.release()
    out.release()
    print(f"🎉 视频分析完成！结果已导出至: {output_path}")

In [12]:
# 定义输入与输出文件路径
input_video_path = 'C0854.mp4'
output_video_path = 'output_yolo_mediapipe_perfect.mp4'

# 执行分析流程
process_badminton_video(input_video_path, output_video_path, yolo_model, options)

🎬 开始处理视频: C0854.mp4
🎉 视频分析完成！结果已导出至: output_yolo_mediapipe_perfect.mp4
